# Neural Identifier Training with Particle Filters - Lorenz Attractor

In [46]:
# Neural Identifier Training with Particle Filters - Lorenz System (Chaotic)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# 1) True nonlinear system (Lorenz Attractor)
# ============================================================
def plant_dynamics(x_state, u=0):
    """
    Continuous dynamics for Lorenz System.
    x_state = [x, y, z]
    
    Equations:
    dx/dt = sigma * (y - x)
    dy/dt = x * (rho - z) - y
    dz/dt = x * y - beta * z
    
    Standard Chaotic Params: sigma=10, rho=28, beta=8/3
    """
    sigma = 10.0
    rho = 28.0
    beta = 8.0 / 3.0
    
    x, y, z = x_state
    
    dx_dt = sigma * (y - x)
    dy_dt = x * (rho - z) - y
    dz_dt = x * y - beta * z
    
    return np.array([dx_dt, dy_dt, dz_dt])

def plant(x_k, u_k, dt=0.01, process_noise_type='laplacian', process_noise_std=1e-2):
    """
    One Euler step of the discrete plant with process noise.
    For Lorenz, RK4 is usually better, but for small dt Euler works for demo.
    """
    # Simple Euler integration
    x_dot = plant_dynamics(x_k)
    x_kp1 = x_k + dt * x_dot

    # Add process noise
    if process_noise_type == 'laplacian':
        noise = np.random.laplace(0, process_noise_std, size=x_kp1.shape)
    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std
        noise = np.random.uniform(-a, a, size=x_kp1.shape)
    else:  # gaussian
        noise = np.random.normal(0, process_noise_std, size=x_kp1.shape)

    return x_kp1 + noise

# ============================================================
# 2) RHONN structure (Updated for 3 Inputs)
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    z = np.clip(z, -100, 100) # Clip to avoid overflow
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est):
    """
    Features for 3-state Lorenz System.
    The system has terms like x*y and x*z.
    We include pairwise cross-products of sigmoids to capture this.
    
    x_est = [x, y, z]
    """
    # Scaling inputs slightly helps sigmoid not saturate immediately for Lorenz values (which go up to ~40)
    scale = 0.1 
    s1 = sigmoidal(x_est[0] * scale)
    s2 = sigmoidal(x_est[1] * scale)
    s3 = sigmoidal(x_est[2] * scale)
    
    return np.array([
        s1, s2, s3,                   # First order sigmoids
        s1*s2, s1*s3, s2*s3,          # Second order interactions (Important for Lorenz!)
        s1**2, s2**2, s3**2,          # Quadratic self-terms
        x_est[0], x_est[1], x_est[2], # Linear terms (essential for the -x, -y, -bz parts)
        1.0                           # Bias
    ])


def RHONN_predict(x_state_for_z, w_neuron):
    """
    Predicts next-state component with a single RHONN neuron.
    """
    z_i = construct_z_vector(x_state_for_z)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

# ============================================================
# 3) EKF trainer over weights
# ============================================================
class EKF_RHONN_Trainer:
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta
        self.weights, self.P, self.Q, self.R = [], [], [], []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.05
            self.weights.append(w_i)
            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, x_hat_previous):
        # Series-Parallel: Use measured true state (chi_k) to build feature vector
        x_state_for_z = np.copy(chi_k) 

        z_i = construct_z_vector(x_state_for_z)          
        H_i = z_i.reshape(-1, 1)                          

        for i in range(self.num_neurons):
            P_pred = self.P[i] + self.Q[i] + np.eye(self.num_weights_per_neuron) * 1e-8
            
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-10: M_i = 1e-10

            x_hat_pred_i = self.weights[i] @ z_i
            e_i = chi_kp1[i] - x_hat_pred_i
            e_i = np.clip(e_i, -50.0, 50.0) # Larger clip for Lorenz

            K_i = (P_pred @ H_i).flatten() / M_i

            adaptive_eta = self.eta * (1.0 / (1.0 + np.abs(e_i) * 0.01))
            self.weights[i] += adaptive_eta * K_i * e_i

            I_KH = np.eye(self.num_weights_per_neuron) - np.outer(K_i, H_i.ravel())
            self.P[i] = I_KH @ P_pred @ I_KH.T + np.outer(K_i, K_i) * self.R[i][0]
            
            # PSD enforcement
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            if np.min(np.linalg.eigvals(self.P[i])) <= 0:
                self.P[i] += np.eye(self.num_weights_per_neuron) * 1e-5

# ============================================================
# 4) Particle Filter trainer over weights
# ============================================================
class PF_RHONN_Trainer:
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=500,
                 initial_weights=None, Q_std=0.5, R_std=0.5, ess_threshold=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        self.Q_std = Q_std
        self.R_var = R_std**2
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        self.particles = []
        self.weights_pf = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                base = np.copy(initial_weights[i])
                # Initialize cloud around base
                particles_i = base + np.random.randn(n_particles, num_weights_per_neuron) * 0.05
            else:
                particles_i = np.random.randn(n_particles, num_weights_per_neuron) * 0.05
            self.particles.append(particles_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)

    def _ess(self, w):
        sum_w = np.sum(w)
        if sum_w == 0: return 0
        w_norm = w / sum_w
        return 1.0 / np.sum(w_norm**2)

    def _resample_systematic(self, neuron_index):
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]
        w = w / np.sum(w)
        N = len(w)
        u0 = np.random.uniform(0.0, 1.0 / N)
        cdf = np.cumsum(w)
        indexes = np.zeros(N, dtype=int)
        i, j = 0, 0
        while i < N:
            u = u0 + i / N
            while u > cdf[j] and j < N - 1:
                j += 1
            indexes[i] = j
            i += 1
        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N

    def update(self, chi_kp1, chi_k, x_hat_previous):
        x_state_for_z = np.copy(chi_k) # Series-Parallel
        z = construct_z_vector(x_state_for_z)

        for i in range(self.num_neurons):
            # Predict (Drift/Diffusion of weights)
            self.particles[i] += np.random.randn(self.n_particles, self.num_weights_per_neuron) * self.Q_std

            # Update (Likelihood)
            w_mat = self.particles[i]
            x_pred_particles = w_mat @ z
            innov = chi_kp1[i] - x_pred_particles
            
            # Robust likelihood
            ll = -0.5 * (innov**2) / self.R_var
            ll -= np.max(ll) # Numerical stability
            like = np.exp(ll) + 1e-300
            
            self.weights_pf[i] *= like
            self.weights_pf[i] /= np.sum(self.weights_pf[i])

            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_systematic(i)

    def get_estimate(self):
        return [np.average(self.particles[i], weights=self.weights_pf[i], axis=0) for i in range(self.num_neurons)]


# ============================================================
# 4b) UKF trainer
# ============================================================
class UKF_RHONN_Trainer:
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0, 
                 alpha=1e-3, beta=2.0, kappa=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta
        self.alpha, self.beta = alpha, beta
        self.kappa = kappa if kappa is not None else 3 - num_weights_per_neuron
        self.n = num_weights_per_neuron
        self.lambda_ = alpha**2 * (self.n + self.kappa) - self.n
        
        # Sigma point weights
        self.Wm = np.zeros(2 * self.n + 1)
        self.Wc = np.zeros(2 * self.n + 1)
        self.Wm[0] = self.lambda_ / (self.n + self.lambda_)
        self.Wc[0] = self.lambda_ / (self.n + self.lambda_) + (1 - alpha**2 + beta)
        for i in range(1, 2 * self.n + 1):
            self.Wm[i] = 1.0 / (2 * (self.n + self.lambda_))
            self.Wc[i] = 1.0 / (2 * (self.n + self.lambda_))

        self.weights, self.P, self.Q, self.R = [], [], [], []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.05
            self.weights.append(w_i)
            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def _generate_sigma_points(self, mean, covariance):
        n = len(mean)
        sigma_points = np.zeros((2 * n + 1, n))
        sigma_points[0] = mean
        try:
            sqrt = np.linalg.cholesky((n + self.lambda_) * covariance)
        except np.linalg.LinAlgError:
            U, s, Vh = np.linalg.svd(covariance)
            sqrt = U @ np.diag(np.sqrt(s)) @ Vh
            sqrt *= np.sqrt(n + self.lambda_)
        
        for i in range(n):
            sigma_points[i + 1] = mean + sqrt[i]
            sigma_points[i + 1 + n] = mean - sqrt[i]
        return sigma_points

    def update(self, chi_kp1, chi_k, x_hat_previous):
        x_state_for_z = np.copy(chi_k) # SP
        z_i = construct_z_vector(x_state_for_z)

        for i in range(self.num_neurons):
            sigma_points = self._generate_sigma_points(self.weights[i], self.P[i])
            
            # Predict Mean/Cov (Static weights assumption in predict step for parameter estimation)
            predicted_mean = np.sum(self.Wm[:, np.newaxis] * sigma_points, axis=0)
            predicted_cov = self.Q[i].copy()
            for j in range(2 * self.n + 1):
                diff = sigma_points[j] - predicted_mean
                predicted_cov += self.Wc[j] * np.outer(diff, diff)
            
            # Measurement (Neural Net output)
            meas_sigmas = np.dot(sigma_points, z_i) # (2n+1,)
            pred_meas = np.sum(self.Wm * meas_sigmas)
            
            innov_cov = self.R[i][0]
            cross_cov = np.zeros(self.n)
            
            for j in range(2 * self.n + 1):
                diff_meas = meas_sigmas[j] - pred_meas
                diff_state = sigma_points[j] - predicted_mean
                innov_cov += self.Wc[j] * diff_meas**2
                cross_cov += self.Wc[j] * diff_state * diff_meas
            
            if innov_cov < 1e-12: innov_cov = 1e-12
            K = cross_cov / innov_cov
            
            innov = chi_kp1[i] - pred_meas
            
            self.weights[i] = predicted_mean + self.eta * K * innov
            self.P[i] = predicted_cov - np.outer(K, K) * innov_cov
            
            self.P[i] = 0.5*(self.P[i] + self.P[i].T) + np.eye(self.n)*1e-6

# ============================================================
# 5) Simulation
# ============================================================
if __name__ == "__main__":
    # --- Simulation settings ---
    n_steps = 2000 # More steps for Lorenz to see the attractor
    dt = 0.01
    t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

    process_noise_type = 'laplacian'    # 
    process_noise_std = 0.5 
    measurement_noise_std = 0.1

    # --- True system init ---
    x_true = np.zeros((n_steps, 3))
    x_true[0] = [1.0, 1.0, 1.0] # Initial condition off-center
    y_measured = np.zeros((n_steps, 3))
    
    # --- RHONN config ---
    num_neurons = 3    # x, y, z
    num_features = 13  # Defined in construct_z_vector (3 sig, 3 cross, 3 quad, 3 lin, 1 bias)
    num_weights_per_neuron = num_features

    # --- Initial weights ---
    common_initial_weights = [np.random.uniform(-0.1, 0.1, num_weights_per_neuron) for _ in range(num_neurons)]

    # --- Instantiate Trainers ---
    ekf_trainer = EKF_RHONN_Trainer(num_neurons, num_weights_per_neuron, initial_weights=common_initial_weights, eta=1.0, Q_init=1e-4)
    ukf_trainer = UKF_RHONN_Trainer(num_neurons, num_weights_per_neuron, initial_weights=common_initial_weights, eta=1.0, Q_init=1e-4)
    pf_trainer = PF_RHONN_Trainer(num_neurons, num_weights_per_neuron, n_particles=500, initial_weights=common_initial_weights, Q_std=0.1, R_std=0.5)

    # Initialize Particle Filter cloud
    for i in range(num_neurons):
        pf_trainer.particles[i] = np.tile(common_initial_weights[i], (pf_trainer.n_particles, 1))
        pf_trainer.particles[i] += np.random.randn(pf_trainer.n_particles, num_weights_per_neuron) * 0.05

    # Storage
    x_hat_ekf = np.zeros((n_steps, 3))
    x_hat_ekf[0] = x_true[0]
    x_hat_ukf = np.zeros((n_steps, 3))
    x_hat_ukf[0] = x_true[0]
    x_hat_pf = np.zeros((n_steps, 3))
    x_hat_pf[0] = x_true[0]

    print("Starting Lorenz Attractor simulation...")
    for k in range(n_steps - 1):
        # 1) Evolve true system
        x_true[k+1] = plant(x_true[k], 0, dt, process_noise_type, process_noise_std)
        y_measured[k+1] = x_true[k+1] + np.random.normal(measurement_noise_std)

        # 2) EKF
        ekf_trainer.update(x_true[k+1], x_true[k], x_hat_ekf[k])
        # Prediction for next step (Series-Parallel)
        z_ekf = construct_z_vector(x_true[k]) 
        for i in range(3): x_hat_ekf[k+1, i] = np.dot(ekf_trainer.weights[i], z_ekf)

        # 3) UKF
        ukf_trainer.update(x_true[k+1], x_true[k], x_hat_ukf[k])
        z_ukf = construct_z_vector(x_true[k])
        for i in range(3): x_hat_ukf[k+1, i] = np.dot(ukf_trainer.weights[i], z_ukf)

        # 4) PF
        pf_trainer.update(x_true[k+1], x_true[k], x_hat_pf[k])
        est_w_pf = pf_trainer.get_estimate()
        z_pf = construct_z_vector(x_true[k])
        for i in range(3): x_hat_pf[k+1, i] = np.dot(est_w_pf[i], z_pf)

        if k % 200 == 0:
            print(f"Step {k}/{n_steps}")

    print("Simulación Completa.")


Starting Lorenz Attractor simulation...
Step 0/2000
Step 200/2000
Step 400/2000
Step 600/2000
Step 800/2000
Step 1000/2000
Step 1200/2000
Step 1400/2000
Step 1600/2000
Step 1800/2000
Simulación Completa.


In [47]:

    # ============================================================
    # 6) Visualización - Formato Tesis
    # ============================================================
    
    # Configuración de formato para tesis
    thesis_config = {
        'font_family': 'Computer Modern, serif',
        'font_size': 14,
        'title_font_size': 16,
        'legend_font_size': 12,
        'line_width_true': 2.5,
        'line_width_est': 2.0,
        'plot_width': 1000,
        'plot_height': 500,
        'grid_color': 'rgba(200, 200, 200, 0.3)',
        'grid_width': 0.5
    }
    
    # Cálculo de MSE por estado
    mse_x_ekf = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)
    mse_y_ekf = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)
    mse_z_ekf = np.mean((x_true[:, 2] - x_hat_ekf[:, 2])**2)
    mse_x_ukf = np.mean((x_true[:, 0] - x_hat_ukf[:, 0])**2)
    mse_y_ukf = np.mean((x_true[:, 1] - x_hat_ukf[:, 1])**2)
    mse_z_ukf = np.mean((x_true[:, 2] - x_hat_ukf[:, 2])**2)
    mse_x_pf = np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2)
    mse_y_pf = np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2)
    mse_z_pf = np.mean((x_true[:, 2] - x_hat_pf[:, 2])**2)
    
    mse_total_ekf = mse_x_ekf + mse_y_ekf + mse_z_ekf
    mse_total_ukf = mse_x_ukf + mse_y_ukf + mse_z_ukf
    mse_total_pf = mse_x_pf + mse_y_pf + mse_z_pf
    
    # Reporte MSE
    print("\n" + "="*70)
    print("🏆 MEJOR FILTRO: ", end="")
    mse_dict = {'EKF': mse_total_ekf, 'UKF': mse_total_ukf, 'PF': mse_total_pf}
    best_filter = min(mse_dict, key=mse_dict.get)
    print(f"{best_filter} (MSE total: {mse_dict[best_filter]:.6f})")
    print("="*70)
    
    print("\n--- Comparación de Desempeño (MSE) - Atractor de Lorenz ---")
    print(f"EKF MSE x:  {mse_x_ekf:.6f}")
    print(f"EKF MSE y:  {mse_y_ekf:.6f}")
    print(f"EKF MSE z:  {mse_z_ekf:.6f}")
    print(f"UKF MSE x:  {mse_x_ukf:.6f}")
    print(f"UKF MSE y:  {mse_y_ukf:.6f}")
    print(f"UKF MSE z:  {mse_z_ukf:.6f}")
    print(f"PF  MSE x:  {mse_x_pf:.6f}")
    print(f"PF  MSE y:  {mse_y_pf:.6f}")
    print(f"PF  MSE z:  {mse_z_pf:.6f}")
    
    # Gráficas individuales por estado
    states_info = [
        {'idx': 0, 'var': 'x', 'desc': '', 'y_label': 'x'},
        {'idx': 1, 'var': 'y', 'desc': '', 'y_label': 'y'},
        {'idx': 2, 'var': 'z', 'desc': '', 'y_label': 'z'}
    ]
    
    for state_info in states_info:
        i = state_info['idx']
        
        fig = go.Figure()
        
        # Estado real (línea negra gruesa)
        fig.add_trace(go.Scatter(
            x=t_history, y=x_true[:, i],
            mode='lines',
            name='Estado Real',
            line=dict(color='#000000', width=thesis_config['line_width_true']),
            showlegend=True
        ))
        
        # Estimación EKF
        fig.add_trace(go.Scatter(
            x=t_history, y=x_hat_ekf[:, i],
            mode='lines',
            name='EKF-RHONN',
            line=dict(color='#1f77b4', width=thesis_config['line_width_est'], dash='dash'),
            showlegend=True
        ))
        
        # Estimación UKF
        fig.add_trace(go.Scatter(
            x=t_history, y=x_hat_ukf[:, i],
            mode='lines',
            name='UKF-RHONN',
            line=dict(color='#2ca02c', width=thesis_config['line_width_est'], dash='dot'),
            showlegend=True
        ))
        
        # Estimación PF
        fig.add_trace(go.Scatter(
            x=t_history, y=x_hat_pf[:, i],
            mode='lines',
            name='PF-RHONN',
            line=dict(color='#d62728', width=thesis_config['line_width_est'], dash='dashdot'),
            showlegend=True
        ))
        
        fig.update_layout(
            title={
                'text': f'Estado {state_info["var"]}{state_info["desc"]} - Atractor de Lorenz',
                'x': 0.5,
                'xanchor': 'center',
                'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
            },
            xaxis_title='Tiempo (s)',
            yaxis_title=state_info['y_label'],
            xaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                gridwidth=thesis_config['grid_width'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True,
                ticks='outside',
                tickwidth=1.5,
                ticklen=5
            ),
            yaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                gridwidth=thesis_config['grid_width'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True,
                ticks='outside',
                tickwidth=1.5,
                ticklen=5
            ),
            legend=dict(
                orientation='h',
                yanchor='bottom',
                y=1.02,
                xanchor='center',
                x=0.5,
                bgcolor='rgba(255, 255, 255, 0)',
                font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
            ),
            font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
            plot_bgcolor='white',
            paper_bgcolor='white',
            width=thesis_config['plot_width'],
            height=thesis_config['plot_height'],
            margin=dict(l=80, r=40, t=100, b=60)
        )
        
        fig.show()
    
    # Espacio de fase 3D (Atractor de Lorenz)
    fig_phase = go.Figure()
    
    fig_phase.add_trace(go.Scatter3d(
        x=x_true[:, 0], y=x_true[:, 1], z=x_true[:, 2],
        mode='lines',
        name='Atractor Real',
        line=dict(color='#000000', width=4)
    ))
    
    fig_phase.add_trace(go.Scatter3d(
        x=x_hat_ekf[:, 0], y=x_hat_ekf[:, 1], z=x_hat_ekf[:, 2],
        mode='lines',
        name='Estimación EKF-RHONN',
        line=dict(color='#1f77b4', width=3)
    ))
    
    fig_phase.add_trace(go.Scatter3d(
        x=x_hat_ukf[:, 0], y=x_hat_ukf[:, 1], z=x_hat_ukf[:, 2],
        mode='lines',
        name='Estimación UKF-RHONN',
        line=dict(color='#2ca02c', width=3)
    ))
    
    fig_phase.add_trace(go.Scatter3d(
        x=x_hat_pf[:, 0], y=x_hat_pf[:, 1], z=x_hat_pf[:, 2],
        mode='lines',
        name='Estimación PF-RHONN',
        line=dict(color='#d62728', width=3)
    ))
    
    fig_phase.update_layout(
        title={
            'text': 'Espacio de Fases 3D - Atractor de Lorenz',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        scene=dict(
            xaxis_title='x',
            yaxis_title='y',
            zaxis_title='z',
            xaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                showline=True,
                linewidth=1.5,
                linecolor='black'
            ),
            yaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                showline=True,
                linewidth=1.5,
                linecolor='black'
            ),
            zaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                showline=True,
                linewidth=1.5,
                linecolor='black'
            )
        ),
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=1.02,
            xanchor='center',
            x=0.5,
            bgcolor='rgba(255, 255, 255, 0)',
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        width=thesis_config['plot_width'],
        height=thesis_config['plot_width'],  # Aspecto cuadrado
        margin=dict(l=80, r=40, t=100, b=60)
    )
    
    fig_phase.show()
    
    # Gráfica de barras comparando MSE
    fig_mse = go.Figure()
    
    filters = ['EKF-RHONN', 'UKF-RHONN', 'PF-RHONN']
    
    fig_mse.add_trace(go.Bar(
        name='Estado x',
        x=filters,
        y=[mse_x_ekf, mse_x_ukf, mse_x_pf],
        marker_color='#636EFA',
        text=[f'{mse_x_ekf:.2e}', f'{mse_x_ukf:.2e}', f'{mse_x_pf:.2e}'],
        textposition='outside'
    ))
    
    fig_mse.add_trace(go.Bar(
        name='Estado y',
        x=filters,
        y=[mse_y_ekf, mse_y_ukf, mse_y_pf],
        marker_color='#EF553B',
        text=[f'{mse_y_ekf:.2e}', f'{mse_y_ukf:.2e}', f'{mse_y_pf:.2e}'],
        textposition='outside'
    ))
    
    fig_mse.add_trace(go.Bar(
        name='Estado z',
        x=filters,
        y=[mse_z_ekf, mse_z_ukf, mse_z_pf],
        marker_color='#00CC96',
        text=[f'{mse_z_ekf:.2e}', f'{mse_z_ukf:.2e}', f'{mse_z_pf:.2e}'],
        textposition='outside'
    ))
    
    fig_mse.update_layout(
        title={
            'text': 'Comparación de Error Cuadrático Medio (MSE)',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tipo de Filtro',
        yaxis_title='Error Cuadrático Medio (MSE)',
        yaxis=dict(
            type='log',
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        xaxis=dict(
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        barmode='group',
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )
    
    fig_mse.show()


    


🏆 MEJOR FILTRO: PF (MSE total: 0.012177)

--- Comparación de Desempeño (MSE) - Atractor de Lorenz ---
EKF MSE x:  0.012407
EKF MSE y:  0.012900
EKF MSE z:  0.013473
UKF MSE x:  0.008045
UKF MSE y:  0.006697
UKF MSE z:  0.018466
PF  MSE x:  0.004376
PF  MSE y:  0.003974
PF  MSE z:  0.003826


In [48]:
# Resumen del experimento y resultados
report = f"""
## Informe de Identificación RHONN + Filtros (Atractor de Lorenz)

**Configuración del sistema**
- Integración Euler con dt = {dt}
- Pasos simulados: {n_steps}
- Arquitectura RHONN: {num_neurons} neuronas (una por estado), {num_weights_per_neuron} pesos por neurona
- Vector de características (13):
    - 3 sigmoides escaladas, 3 productos cruzados, 3 términos cuadráticos,
        3 términos lineales y 1 sesgo.

**Ruido**
- Ruido de proceso: tipo `{process_noise_type}` con desviación estándar {process_noise_std}
- Ruido de medición: Gaussiano con desviación estándar {measurement_noise_std}

**Entrenadores**
- EKF-RHONN, UKF-RHONN y PF-RHONN
- Pesos iniciales compartidos (`common_initial_weights`) perturbados según cada filtro

**Desempeño (MSE por estado)**
- EKF:   x={mse_x_ekf:.4e}, y={mse_y_ekf:.4e}, z={mse_z_ekf:.4e}, total={mse_total_ekf:.4e}
- UKF:   x={mse_x_ukf:.4e}, y={mse_y_ukf:.4e}, z={mse_z_ukf:.4e}, total={mse_total_ukf:.4e}
- PF:    x={mse_x_pf:.4e}, y={mse_y_pf:.4e}, z={mse_z_pf:.4e}, total={mse_total_pf:.4e}

**Mejor filtro (MSE total más bajo)**: {best_filter}
"""

print(report)


## Informe de Identificación RHONN + Filtros (Atractor de Lorenz)

**Configuración del sistema**
- Integración Euler con dt = 0.01
- Pasos simulados: 2000
- Arquitectura RHONN: 3 neuronas (una por estado), 13 pesos por neurona
- Vector de características (13):
    - 3 sigmoides escaladas, 3 productos cruzados, 3 términos cuadráticos,
        3 términos lineales y 1 sesgo.

**Ruido**
- Ruido de proceso: tipo `laplacian` con desviación estándar 0.5
- Ruido de medición: Gaussiano con desviación estándar 0.1

**Entrenadores**
- EKF-RHONN, UKF-RHONN y PF-RHONN
- Pesos iniciales compartidos (`common_initial_weights`) perturbados según cada filtro

**Desempeño (MSE por estado)**
- EKF:   x=1.2407e-02, y=1.2900e-02, z=1.3473e-02, total=3.8780e-02
- UKF:   x=8.0452e-03, y=6.6969e-03, z=1.8466e-02, total=3.3208e-02
- PF:    x=4.3762e-03, y=3.9742e-03, z=3.8264e-03, total=1.2177e-02

**Mejor filtro (MSE total más bajo)**: PF



In [ ]:

print("="*80)
print("COMPARISON OF INITIAL vs FINAL WEIGHTS")
print("="*80)

for neuron_idx in range(num_neurons):
    print(f"\n{'='*80}")
    print(f"NEURON {neuron_idx} (State Variable: {['x', 'y', 'z'][neuron_idx]})")
    print(f"{'='*80}")
    
    initial_w = common_initial_weights[neuron_idx]
    
    print(f"\n--- INITIAL WEIGHTS ---")
    print(f"Values: {initial_w}")
    print(f"Norm: {np.linalg.norm(initial_w):.6f}")
    
    print(f"\n--- EKF-RHONN FINAL WEIGHTS ---")
    ekf_w = ekf_trainer.weights[neuron_idx]
    print(f"Values: {ekf_w}")
    print(f"Norm: {np.linalg.norm(ekf_w):.6f}")
    print(f"Change (L2): {np.linalg.norm(ekf_w - initial_w):.6f}")
    
    print(f"\n--- UKF-RHONN FINAL WEIGHTS ---")
    ukf_w = ukf_trainer.weights[neuron_idx]
    print(f"Values: {ukf_w}")
    print(f"Norm: {np.linalg.norm(ukf_w):.6f}")
    print(f"Change (L2): {np.linalg.norm(ukf_w - initial_w):.6f}")
    
    print(f"\n--- PF-RHONN FINAL WEIGHTS ---")
    pf_w = est_w_pf[neuron_idx]
    print(f"Values: {pf_w}")
    print(f"Norm: {np.linalg.norm(pf_w):.6f}")
    print(f"Change (L2): {np.linalg.norm(pf_w - initial_w):.6f}")
    
    print(f"\n--- WEIGHT CHANGES COMPARISON ---")
    ekf_change = np.linalg.norm(ekf_w - initial_w)
    ukf_change = np.linalg.norm(ukf_w - initial_w)
    pf_change = np.linalg.norm(pf_w - initial_w)
    
    print(f"EKF change: {ekf_change:.6f}")
    print(f"UKF change: {ukf_change:.6f}")
    print(f"PF change:  {pf_change:.6f}")
    print(f"Max change: {max(ekf_change, ukf_change, pf_change):.6f} ({['EKF', 'UKF', 'PF'][[ekf_change, ukf_change, pf_change].index(max(ekf_change, ukf_change, pf_change))]})")